# Chapter 2: Generating Text with a Pre-trained LLM

In [1]:
from importlib.metadata import version

used_libraries = [
    "reasoning_from_scratch",
    "torch",
    "tokenizers"  # Used by reasoning_from_scratch
]

for lib in used_libraries:
    print(f"{lib} version: {version(lib)}")

reasoning_from_scratch version: 0.1.21
torch version: 2.10.0
tokenizers version: 0.22.2


In [2]:
!pip install -r https://raw.githubusercontent.com/rasbt/reasoning-from-scratch/refs/heads/main/requirements.txt --quiet

In [3]:
# Dependencies can also be installed manually:
#!pip install torch>=2.10.0 tokenizers>=0.22.2 reasoning-from-scratch

### UV installation & startup
- run the installation for your OS from the official website: https://docs.astral.sh/uv/getting-started/installation/

In [4]:
!uv --version

uv 0.10.7


- (terminal) navigate to `reasoning-from-scratch`
- Run `uv run jupyter lab` to launch JupyterLab and open a blank notebook or the notebook for this chapter
- It creates a local virtual environment (usually in `.venv/`) and installs dependencies from `pyproject.toml`.

## 2.3 Hardware

- If new to PyTorch: [PyTorch in One Hour](https://sebastianraschka.com/teaching/pytorch-1h/) tutorial
- If you followed the previous section, you should have PyTorch installed
- Check your local PyTorch installation supports a GPU

In [5]:
import torch

print(f"PyTorch version {torch.__version__}")

if torch.cuda.is_available():
    print(f"CUDA/ROCm GPU: {torch.cuda.get_device_name(0)}")

elif torch.xpu.is_available():
    print(f"Intel GPU: {torch.xpu.get_device_name(0)}")

elif torch.backends.mps.is_available():
    print("Apple Silicon GPU")

else:
    print("Only CPU")

PyTorch version 2.10.0+cu128
CUDA/ROCm GPU: NVIDIA GeForce RTX 3060 Laptop GPU


- Most code will use an NVIDIA (CUDA) GPU if available, otherwise run on a CPU (or Apple Silicon GPU)
- Chapters 2-4 can be executed in a reasonable time on a CPU
- Chapters 5-7 will be slow on a CPU - a GPU with CUDA support is recommended.
- [Lightning AI Studio](https://lightning.ai/) offers free compute credits after sign-up & verification
- [Google Colab](https://colab.research.google.com/) is another good choice
  See [../02_setup-tips/gpu-instructions.md](../02_setup-tips/gpu-instructions.md) for cloud compute tips

## 2.4 Tokenizers

- tokenizers encode input text into a token IDs for LLMs
- tokenizer also decode LLM outputs back to human-readable text

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch02/CH02_F05_raschka.webp?1" width="500px">

- building an LLM and tokenizer from scratch is outside the scope of this book
- we will use a pre-trained LLM in the next section; here, we load the tokenizer that goes with it
- The `reasoning_from_scratch` Python package provides the base LLM and corresponding tokenizer
- Tokenizer designed using the [`tokenizers`](https://github.com/huggingface/tokenizers) Python package
- `reasoning_from_scratch` is part of this book's supplementary code - it should already be installed based on the instructions in section 2.2

In [7]:
# download tokenizer setup files for Qwen3
from reasoning_from_scratch.qwen3 import download_qwen3_small
download_qwen3_small(kind="base", tokenizer_only=True, out_dir="qwen3")
!ls qwen3

tokenizer-base.json


In [8]:
# load tokenizer settings into Qwen3Tokenizer
from pathlib import Path
from reasoning_from_scratch.qwen3 import Qwen3Tokenizer

tokenizer_path = Path("qwen3") / "tokenizer-base.json"
tokenizer = Qwen3Tokenizer(tokenizer_file_path=tokenizer_path)

#### Since we haven't loaded the LLM itself yet, we will do a simpler round-trip

In [11]:
prompt = "Explain large language models."

In [12]:
# encode prompt into tokens
input_token_ids_list = tokenizer.encode(prompt)
for i in input_token_ids_list:
    print(f"{i} --> {tokenizer.decode([i])}")

840 --> Ex
20772 --> plain
3460 -->  large
4128 -->  language
4119 -->  models
13 --> .


In [13]:
# decode tokens back to text
text = tokenizer.decode(input_token_ids_list); print(text)

Explain large language models.


- FYI: `Qwen3Tokenizer` has ~151K unique tokens (vocabulary size)

- Additional resources on tokenization:
  - [Build a Large Language Model (from Scratch)](https://mng.bz/M96o) chapter 2
  - [Implementing A Byte Pair Encoding (BPE) Tokenizer From Scratch](https://sebastianraschka.com/blog/2025/bpe-from-scratch.html)

## 2.5 Loading pre-trained models

- This book uses Qwen3 0.6B.
  - Qwen3 is the leading open-weight model in terms of modeling performance as of this writing
  - Qwen3 0.6B is more memory efficient than Llama 3 1B
  - There's both a base model (which we focus on for reasoning model development) and an official reasoning variant that we can use as a reference model
- We are using Qwen3 rewritten in pure PyTorch without any external LLM library dependencies
- See appendix C for the Qwen3 model code
- See appendix D for loading the reasoning variant and larger Qwen3 models
- See the Qwen3 [GitHub repository](https://github.com/QwenLM/Qwen3) and [technical report](https://arxiv.org/abs/2505.09388) for (even) more details

In [14]:
def get_device(enable_tensor_cores=True):
    if torch.cuda.is_available():
        device = torch.device("cuda")
        print("Using NVIDIA CUDA GPU")
        
        if enable_tensor_cores:
            major, minor = map(int, torch.__version__.split(".")[:2])
            if (major, minor) >= (2, 9):
                torch.backends.cuda.matmul.fp32_precision = "tf32"
                torch.backends.cudnn.conv.fp32_precision = "tf32"
            else:
                torch.backends.cuda.matmul.allow_tf32 = True
                torch.backends.cudnn.allow_tf32 = True

    elif torch.backends.mps.is_available():
        device = torch.device("mps")
        print("Using Apple Silicon GPU (MPS)")

    elif torch.xpu.is_available():
        device = torch.device("xpu")
        print("Using Intel GPU")

    else:
        device = torch.device("cpu")
        print("Using CPU")

    return device

device = get_device()

Using NVIDIA CUDA GPU


In [15]:
# Recommended: Use CPU on the first run-through
device = torch.device("cpu")

In [16]:
# download pretrained model weights, ~1.5GB
download_qwen3_small(kind="base", tokenizer_only=False, out_dir="qwen3")

qwen3-0.6B-base.pth: 100% (1433 MiB / 1433 MiB)


In [17]:
from reasoning_from_scratch.qwen3 import Qwen3Model, QWEN_CONFIG_06_B

model_path = Path("qwen3") / "qwen3-0.6B-base.pth"
model      = Qwen3Model(QWEN_CONFIG_06_B)

model.load_state_dict(torch.load(model_path))
model.to(device)

Qwen3Model(
  (tok_emb): Embedding(151936, 1024)
  (trf_blocks): ModuleList(
    (0-27): 28 x TransformerBlock(
      (att): GroupedQueryAttention(
        (W_query): Linear(in_features=1024, out_features=2048, bias=False)
        (W_key): Linear(in_features=1024, out_features=1024, bias=False)
        (W_value): Linear(in_features=1024, out_features=1024, bias=False)
        (out_proj): Linear(in_features=2048, out_features=1024, bias=False)
        (q_norm): RMSNorm()
        (k_norm): RMSNorm()
      )
      (ff): FeedForward(
        (fc1): Linear(in_features=1024, out_features=3072, bias=False)
        (fc2): Linear(in_features=1024, out_features=3072, bias=False)
        (fc3): Linear(in_features=3072, out_features=1024, bias=False)
      )
      (norm1): RMSNorm()
      (norm2): RMSNorm()
    )
  )
  (final_norm): RMSNorm()
  (out_head): Linear(in_features=1024, out_features=151936, bias=False)
)

## 2.6 LLM text generation

In [18]:
example = torch.tensor([1, 2, 3]) 
print(example)
print(example.unsqueeze(0)) # adds new dimension of size 1 at index 0.

tensor([1, 2, 3])
tensor([[1, 2, 3]])


In [19]:
example = torch.tensor([[1, 2, 3]]) 
print(example)
print(example.squeeze(0)) # removes dimension at index 0, if it has a size of 1.

tensor([[1, 2, 3]])
tensor([1, 2, 3])


In [20]:
prompt = "Explain large language models."

input_token_ids_list = tokenizer.encode(prompt)
input_tensor         = torch.tensor(input_token_ids_list)
input_tensor_fmt     = input_tensor.unsqueeze(0).to(device)

print(f"#input tokens: {len(input_token_ids_list)}")

#input tokens: 6


- `torch.inference_mode()` disables gradient tracking & view tracking during inference to improve speed and reduce memory.
- similar to `torch.no_grad()` but more aggressive—it also disables internal version counters and view tracking that are only needed during training. Use it for inference when you don't need gradients.

In [21]:
with torch.inference_mode():
    output_tensor = model(input_tensor_fmt)

output_tensor_fmt = output_tensor.squeeze(0)
print(f"Formatted Output tensor shape: {output_tensor_fmt.shape}")

Formatted Output tensor shape: torch.Size([6, 151936])


- 6 input tokens returns a 6x151K matrix (#tokens x vocab size)
<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch02/CH02_F12_raschka.webp" width="500px">

In [24]:
# extract last row of output matrix
# bfloat16 = reduced-precision format = better efficiency
last_token = output_tensor_fmt[-1]
print(last_token)

tensor([ 7.3125,  1.9453,  7.9062,  ..., -2.4688, -2.4688, -2.4688],
       dtype=torch.bfloat16)


In [29]:
# argmax = returns position with largest score in a tensor
# (compare to torch.max which returns largest value in a tensor
# keepdim=True keeps output shape consistent by retaining reduced dimensions
print(torch.argmax(last_token, dim=-1, keepdim=True))

tensor([20286])


In [30]:
# decoded text string at that location
print(tokenizer.decode([20286]))

 Large


In [31]:
example = torch.tensor([-2, 1, 3, 1])
print(torch.max(example))
print(torch.argmax(example))

tensor(3)
tensor(2)


## 2.7 minimal text generator
`generate_text_basic_stream`

In [32]:
@torch.inference_mode()

def generate_text_basic_stream(
    model,
    token_ids,
    max_new_tokens, 
    eos_token_id=None):
    model.eval()

    for _ in range(max_new_tokens):
        out = model(token_ids)[:, -1]
        next_token = torch.argmax(out, dim=-1, keepdim=True)

        # Stop if we encounter an end-of-sequence token
        if (eos_token_id is not None
                and torch.all(next_token == eos_token_id)):
            break

        yield next_token  # Yield each token as it's generated
        
        token_ids = torch.cat([token_ids, next_token], dim=1)

- Generate a 100-token response to a simple prompt
- The following code will be slow. Improvements to follow.

In [37]:
prompt = "Explain large language models in a single sentence."
#prompt = "Explain american college football in a single sentence."
max_new_tokens = 100

input_token_ids_tensor = torch.tensor(
    tokenizer.encode(prompt),
    device=device).unsqueeze(0)

for token in generate_text_basic_stream(
    model          = model,
    token_ids      = input_token_ids_tensor,
    max_new_tokens = max_new_tokens):
    token_id       = token.squeeze(0).tolist()
    print(
        tokenizer.decode(token_id),
        end="",
        flush=True  # Deactivates buffering so tokens are printed live
    )

 Large language models are artificial intelligence systems that can understand, generate, and process human language, enabling them to perform a wide range of tasks, from answering questions to writing articles, and even creating creative content.<|endoftext|>Human language is a complex and dynamic system that has evolved over millions of years to enable effective communication and social interaction. It is composed of a vast array of symbols, including letters, numbers, and symbols, which are used to convey meaning and express thoughts and ideas. The evolution of language has

- The LLM response becomes nonsensical/off-topic after `<|endoftext|>`, which is a token used as a delimiter between different documents during training
- we want it to stop generating after encountering this token.

In [38]:
# <|endoftext|> is stored as a tokenizer attribute.
print(tokenizer.encode("<|endoftext|>"))
print(tokenizer.eos_token_id)

[151643]
151643


In [41]:
# use it to stop generating decoded text
for token in generate_text_basic_stream(
    model=model,
    token_ids=input_token_ids_tensor,
    max_new_tokens=max_new_tokens,
    eos_token_id=tokenizer.eos_token_id
):
    token_id = token.squeeze(0).tolist()
    print(
        tokenizer.decode(token_id),
        end="",
        flush=True
    )

 Large language models are artificial intelligence systems that can understand, generate, and process human language, enabling them to perform a wide range of tasks, from answering questions to writing articles, and even creating creative content.

In [43]:
# simple benchmarking function
# returns total runtime, token generation speed & GPU memory usage
import warnings

def generate_stats(output_token_ids, tokenizer, start_time,
                   end_time):
    total_time = end_time - start_time
    print(f"\n\nTime: {total_time:.2f} sec")
    print(f"{int(output_token_ids.numel() / total_time)} tokens/sec")

    for name, backend in (("CUDA", getattr(torch, "cuda", None)),
                          ("XPU", getattr(torch, "xpu", None))):
        if backend is not None and backend.is_available():

            # Check whether we are actually using this backend
            device_type = output_token_ids.device.type
            if device_type != name.lower():
                warnings.warn(
                    f"{name} is available but tensors are on "
                    f"{device_type}. Memory stats may be 0."
                )
    
            # Synchronize if supported (important for async backends)
            if hasattr(backend, "synchronize"):
                backend.synchronize()
            
            max_mem_bytes = backend.max_memory_allocated()
            max_mem_gb = max_mem_bytes / (1024 ** 3)
            print(f"Max {name} memory allocated: {max_mem_gb:.2f} GB")
            backend.reset_peak_memory_stats()

In [52]:
# suppress CUDA warnings while running solely on CPU
import warnings
warnings.filterwarnings('ignore', 
                        message='CUDA is available but tensors are on cpu')


In [53]:
import time

start_time = time.time()
generated_ids = []

for token in generate_text_basic_stream(
    model=model,
    token_ids=input_token_ids_tensor,
    max_new_tokens=max_new_tokens,
    eos_token_id=tokenizer.eos_token_id
):
    token_id = token.squeeze(0).tolist()
    print(
        tokenizer.decode(token_id),
        end="",
        flush=True
    )

    next_token_id = token.squeeze(0)
    generated_ids.append(next_token_id)  # Collect generated tokens

end_time = time.time()

output_token_ids_tensor = torch.cat(generated_ids, dim=0)
generate_stats(output_token_ids_tensor, tokenizer, start_time, end_time)

 Large language models are artificial intelligence systems that can understand, generate, and process human language, enabling them to perform a wide range of tasks, from answering questions to writing articles, and even creating creative content.

Time: 26.50 sec
1 tokens/sec
Max CUDA memory allocated: 0.00 GB


## 2.8 Faster inference via KV caching
- KV = keys and values inside the attention mechanism of the LLM
- [KV Caches from Scratch](https://magazine.sebastianraschka.com/p/coding-the-kv-cache-in-llms)
- `generate_text_basic_stream`
- Previously, each new token was concatenated to the entire input sequence and fed back to the model repeatedly. This approach is inefficient - all tokens, except the newly generated one, remained identical in subsequent iterations. KV caches avoid redundant computation.

In [54]:
from reasoning_from_scratch.qwen3 import KVCache

@torch.inference_mode()
def generate_text_basic_stream_cache(
    model,
    token_ids,
    max_new_tokens,
    eos_token_id=None
):
    model.eval()
    cache = KVCache(n_layers=model.cfg["n_layers"])  # New
    model.reset_kv_cache()                           # New

    out = model(token_ids, cache=cache)[:, -1]
    for _ in range(max_new_tokens):
        next_token = torch.argmax(out, dim=-1, keepdim=True)

        if (eos_token_id is not None
                and torch.all(next_token == eos_token_id)):
            break

        yield next_token
        out = model(next_token, cache=cache)[:, -1]

- Generating n new tokens without KV caching requires recomputing work over an increasingly long sequence (roughly O(n2) = square of the output length).
- With KV caching, after the initial pass, each new step processes only the newly added token, reducing the work to roughly O(n). The total work grows approximately in direct proportion to the number of generated tokens.

In [55]:
start_time = time.time()
generated_ids = []

for token in generate_text_basic_stream_cache(
    model=model,
    token_ids=input_token_ids_tensor,
    max_new_tokens=max_new_tokens,
    eos_token_id=tokenizer.eos_token_id
):
    token_id = token.squeeze(0).tolist()
    print(
        tokenizer.decode(token_id),
        end="",
        flush=True
    )

    next_token_id = token.squeeze(0)
    generated_ids.append(next_token_id)  # Collect generated tokens

end_time = time.time()

output_token_ids_tensor = torch.cat(generated_ids, dim=0)
generate_stats(output_token_ids_tensor, tokenizer, start_time, end_time)

 Large language models are artificial intelligence systems that can understand, generate, and process human language, enabling them to perform a wide range of tasks, from answering questions to writing articles, and even creating creative content.

Time: 2.78 sec
14 tokens/sec
Max CUDA memory allocated: 0.00 GB


- Much faster execution
- Same output

## 2.9 Faster inference via PyTorch model compilation

- Another technique to speed up the model inference (text generation) by a lot is using `torch.compile`
- The usage is simple, we just call `torch.compile` on the model (see [the documentation](https://docs.pytorch.org/docs/stable/torch.compiler_api.html) for additional options)

In [56]:
# avoid model recompilation triggers in PyTorch 2.8 and newer,
# if model contains code like self.pos = self.pos+1

major, minor = map(int, torch.__version__.split(".")[:2])
if (major, minor) >= (2, 8):
    torch._dynamo.config.allow_unspec_int_on_nn_module = True

model_compiled = torch.compile(model)

# If you have issues with torch.compile on "mps" devices and get an InductorError,
# make sure you are using PyTorch 2.9 or newer|

---

**Windows note 1**

- Compilation can be tricky on Windows
- `torch.compile()` uses Inductor, which JIT-compiles kernels and needs a working C/C++ toolchain
- For CUDA, Inductor also depends on Triton, available via the community package `triton-windows`
  - If you see `cl not found`, [install Visual Studio Build Tools with the "C++ workload"](https://learn.microsoft.com/en-us/cpp/build/vscpp-step-0-installation?view=msvc-170) and run Python from the "x64 Native Tools" prompt
  - If you see `triton not found` with CUDA, install `triton-windows` (for example, `uv pip install "triton-windows<3.4"`).
- For CPU, a reader further recommended following this [PyTorch Inductor guide for Windows](https://docs.pytorch.org/tutorials/unstable/inductor_windows.html)
  - Here, it is important to install the English language package when installing Visual Studio 2022 to avoid a UTF-8 error
  - Also, please note that the code needs to be run via the "Visual Studio 2022 Developer Command Prompt" rather than a notebook
- If this setup proves tricky, you can skip compilation; **compilation is optional, and all code examples work fine without it**

---

**Windows note 2**

- Readers reported that there is no speed-up when running `torch.compile` with default settings on Windows; however, running `torch.compile` with the `"max-autotune"` mode resulted in a 2x speed-up: `torch.compile(model, mode="max-autotune")`

---

In [59]:
# The first iteration will be slow - it does initial compilation and optimization
# hence, we repeat the text generation multiple times
# Start with the non-cached version - SLOW!

for i in range(3):
    start_time = time.time()
    generated_ids = []

    for token in generate_text_basic_stream(
        model=model_compiled,
        token_ids=input_token_ids_tensor,
        max_new_tokens=max_new_tokens,
        eos_token_id=tokenizer.eos_token_id
    ):
        token_id = token.squeeze(0).tolist()
        print(
            tokenizer.decode(token_id),
            end="",
            flush=True)

        next_token_id = token.squeeze(0)
        generated_ids.append(next_token_id)

    end_time = time.time()

    if i == 0:
        print("Warm-up run")
    else:
        print(f"Timed run {i}:")

    output_token_ids_tensor = torch.cat(generated_ids, dim=0)
    generate_stats(output_token_ids_tensor, tokenizer, start_time, end_time)

    print(f"\n{30*'-'}")

 Large language models are artificial intelligence systems that can understand, generate, and process human language, enabling them to perform tasks such as answering questions, writing text, and even creating music.Warm-up run


Time: 19.28 sec
1 tokens/sec
Max CUDA memory allocated: 0.00 GB

------------------------------
 Large language models are artificial intelligence systems that can understand, generate, and process human language, enabling them to perform tasks such as answering questions, writing text, and even creating music.Timed run 1:


Time: 18.53 sec
1 tokens/sec
Max CUDA memory allocated: 0.00 GB

------------------------------
 Large language models are artificial intelligence systems that can understand, generate, and process human language, enabling them to perform tasks such as answering questions, writing text, and even creating music.Timed run 2:


Time: 20.12 sec
1 tokens/sec
Max CUDA memory allocated: 0.00 GB

------------------------------


In [60]:
# KV cache benchmark for comparison
for i in range(3):
    
    start_time = time.time()
    generated_ids = []
    
    for token in generate_text_basic_stream_cache(
        model=model_compiled,
        token_ids=input_token_ids_tensor,
        max_new_tokens=max_new_tokens,
        eos_token_id=tokenizer.eos_token_id
    ):
        token_id = token.squeeze(0).tolist()
        print(
            tokenizer.decode(token_id),
            end="",
            flush=True
        )
    
        next_token_id = token.squeeze(0)
        generated_ids.append(next_token_id)  # Collect generated tokens
    
    end_time = time.time()

    if i == 0:
        print("Warm-up run")
    else:
        print(f"Timed run {i}:")

    output_token_ids_tensor = torch.cat(generated_ids, dim=0)
    generate_stats(
        output_token_ids_tensor, tokenizer, start_time, end_time
    )

    print(f"\n{30*'-'}")

 Large language models are artificial intelligence systems that can understand, generate, and process human language, enabling them to perform a wide range of tasks, from answering questions to writing articles, and even creating creative content.Warm-up run


Time: 1.81 sec
22 tokens/sec
Max CUDA memory allocated: 0.00 GB

------------------------------
 Large language models are artificial intelligence systems that can understand, generate, and process human language, enabling them to perform a wide range of tasks, from answering questions to writing articles, and even creating creative content.Timed run 1:


Time: 1.89 sec
21 tokens/sec
Max CUDA memory allocated: 0.00 GB

------------------------------
 Large language models are artificial intelligence systems that can understand, generate, and process human language, enabling them to perform a wide range of tasks, from answering questions to writing articles, and even creating creative content.Timed run 2:


Time: 1.91 sec
21 token